# Grouping sensitivity runs

Repeats the grouped five-fold cross-validation of EfficientNet-B0 (two-stage, seeds 0–2) under three alternative groupings, with the same code and settings as the main analysis:

* `ncc098_patients` — near-duplicates at NCC ≥ 0.98 plus recovered patient identifiers
* `ncc095_duponly` — near-duplicates at NCC ≥ 0.95, no patient identifiers
* `ncc098_duponly` — near-duplicates at NCC ≥ 0.98, no patient identifiers

**Settings:** Accelerator GPU T4 ×2, Internet ON, input `masoudnickparvar/brain-tumor-mri-dataset`. Output: `/kaggle/working/BPEX_R2_sensitivity_output.zip`.

In [ ]:
import os, sys, json, glob, time, base64, gzip, zipfile, subprocess, threading, queue
T0 = time.time(); WORK = "/kaggle/working"; KIT = f"{WORK}/kit"; OUT = f"{WORK}/results_sensitivity"
os.makedirs(KIT, exist_ok=True); os.makedirs(OUT, exist_ok=True)
TIME_BUDGET_HOURS = 11.0
SMOKE = os.environ.get("BPEX_SMOKE") == "1"
def hours(): return (time.time() - T0) / 3600
BUNDLE = {"common.py": "H4sIAEUo+GoC/71c/3PctrH/XX8FHjN9JZsTfZKtvETNZWo7cuLUdj2WJmnnesPhkeCJEY9kSZ6ks57+9/fZBUCCPJ7ktM7LZHwkASwWi8V+hxzHOb8MKxmLTZNmaZPKWiRFJZpLKZZVmOaHzWZdbCrx9sNrkcnwKlyhQebR5TqsrvyDg7NrWW1FHVVp2Yg0x8C0FlcpntdlUTWAVhVrcSkrKeoCrWEj4rAJRVaEcZqvJmJdxDKrJwcNzaa+yKZKo1qEeSyiIk/S1aYKm7TIBRAVaSzzJo3CTIRRVdS1kLelrNI1vtb+geM4B2pmcRnWl1m6NK+/1kVunovaPJVZ2GC9a/NeYdKifWsA9oAXQDhHWVjXoI9ubD9NRFjHadRMRJLKLD4wo/PNutyiTeRlOx3A4wP+L2MF+P3rNwbg6zWIe3Dw8s3z8/OzczETc2eVpcU6dCbCwfpAHPOWF7QtFT2WabNJm7DaOgs1NHjz/MXZGwy/i05FyruZTkREmyOBkgQtpasn8e71mPcfzi4u/kGDzJynwvlhbHZ8f2vhciAG/7W4oeM77DizTx9RtLxvX+4PgudvXj9XK75jcBYKHQHUU9DCHrbU9kcFpo91n4LdWwdyPVyX9aUe72At1tqTvAge+Lzpf7cBbXaHVOswG3y8lGHWXG5HoYx87BG+e7G3pEPXah+MtnDfA2RNJzLE+dQUt2HdH7x++0Nw9vcL5jH/13JFY/Er1UOZq9/luuTfJk3MbwIeOTj4Qhx+tv8ArC9YXJJ312G2wfGuJJ1GSESWZhJnFvLF+7zz/6UVHgf8r3jJ6JwyydP1KqjTjxKnN29AruPjZ/x9GTbRZa/l6TE3gB/iAFhHl7VpOTox5/ELUTcktI9OBU+VQkhVPEQUebbtAGTVqUgglnm4PHzKLUkzAPx02h11A/r4VNxcFpkUuWxuiurKjLQhnsjDE/5e4wRlMuCRwzmfEcxwmfGmnIq8UHjehNX6cFPy8BJtUD8tCb62Zc8XQoZVBnXUFGWJebBC2tU0VrucQV0wkLgqymLTdHNPfUVJdA6SKozU/F0rk/MLsaqKTXkYAh85EU14JXOl3ohNjP6CwM3iuqMtFrRZkXYynGb6UQ9PkT/J0jIo7QkVrapCjQpiubIINfWn3IzTVgY1VKEM1mkP32+mPLPSZoeVJKaJD6m/+OPHolj/UTFUla4um1zWdW+tGniRA9W6GWmCFglon2XVsoUiX7jGIpZFkeHLRbWRascqyWuWcb9N7YRMRL0urqRbyyzxTlt1AkV+kebYStk0oFYtNrVUHCuaQkSXMrpSpzMtZQbgotrksAVgMqAZPz5ZAgYYwfbNscL8Xz3rN3Uniziq32YdLqJ9v7E9HrtNhlF3WyzyEVUHw1pqoe1VmNWybUfLpsq5G+QhUy68lgEZNm6x/HVCZ+NSk7Co/TX4M06r2sUztfh4ycO1bN/DZU2/Lg/zJjCk0roJiqsZbY/izJu0uRRFKXPuBHl843hkvSSX3U7R/H4MW0chkaBbmpORNjue0P6Gm6yZMZaBfvM0+r2PbqFRTxORQkhAQIB6bjERbl764DK5ktXEs1hE0wNNGLt/KLMvGZYjY7lt72gMhrlWVeF2d2ThN0UGerlqLLasluJiW8qzqioqt8ET4JqVkrWrNsraoz5xd+iqJ2Ly0ng3uTTgainBk2R5w9zOVy69G+ppy7WookuNGckAn7qofuoIl/54Aw8E7+QbiMLRtmgTh3aHIMwyqxO4S+bXaVXkc+f9Py5+/Nu7H5+f/3h+dva9swBL1021A3EZRhClcU2g89xv3YveCRjtGstGVpB92Ig0agUL00gjQZI3SPOkcA2B8Mw2SAniFTnsFOMD+OpLALrWELquR8aNbut10w/o0LN+nbIqIghTNqS63uaj6wmY4u13GEvYO7kDJCo3QVRs8gZAQMv21YWtznSoth2P7Oz28ON1SkvpmrD4ucMtvBuKqIFZchCM9VQwuv7q/YFRxCEWeN2NGWfQc1VurI7MWSvZQCRcp5EMWFpNPTqYVoe0DsLrMM1gJ4B6As6jJJM3l0rey9tIwhU94x9W41CEZRWu1iFbFFEBfFo0ylCbBBpzUJsdUsaqfVOY6JNupE5SaE4bdjsd2adEGKnLUrZ2nSfEGE8wliA5loDpi4Zhv6GUMP+Rm8d6EFbrWLvGgrpAmtxIsKMPSQcPneZyHV6AoAUMcbH/04tnKHWJeIHrnMJMP/LmRwuAQxBAU0lP13J7va0biRMjZjPhfB9WNylOXm8azbX1ZqlPzMHIxF2rzyZAACuu3DTu3MEEUZORy3CYs3+H4xXLks6PvyRRFxB6cDIWHqRGhNUyAT4R41+g0Yqb+jehDNRI3u3F+GadRoQpMGQvVjbs8NEO9HBUZP5n7nhjFJlnZhWKBWj/aWba67aFIind239hQe94mvl0MX5mds+Hnm6fWHM2+VVe3OTO5/fUOFxEFsUttu/zAmcjhFzsgF0jl4ivuf9KbrF79MHQrT02cBCzEBaCc0gbFjjWF6G/2CQz4Q2SbC7AGiXOS1ITB+QzQEq6VVEgjMRbHljIwJZ9Tb3Ft9Thuyff8ij8phQy+o5MoEKE4ntQ6lVFKBvjtwLTUiBJbTMxSAQrkHikZh+XbEGyYvCR57ZNpGSTka1u5NavRZpr/AhG79Ag4ND2S2sCRoMHcoQcijTfdPZslBFuNv13AFMXxBPfQbr3gZUVmX2JI8Qc3li+EPVVqly+TV7h6KxgEsCEVs61ou6puCOs7gfnaActohLwUAZvQEG9jIJ+Oa3wJsyuxtZGYxKLrDxmRI5aeoA3Wd5iESw8NXMRDB0oGZfCtKN+WEI1xC5sGEBSdkK3Ry3uCRkwPA26dDw12SvdLSMko3gX/gUEKFpJESgruDhHy+K+x+Vl7Lfs5xKShs2TlGTvZmlYbMKHyhCHia2J22PEnu6MbfLwcBZpfZYbYdTYG+yBMd7Hulp2/Cts37uieQW7K1b2fMIWhigSccfz3wNzNILbwFnijmAQY9knO0+jqzKsrsPeitqjTM7rX8MVYiHiBTl84oJCcBxnJ0LC7xXuhY4VPOHlXsi64RctLbz2mDcVDtKYPNkl/p1jAhAkqvjZuffaR22Wy98CT6HF4PCoodGTAhYnAAb2wEGL4GzNmwqxE7mAl7jC0ZcBz2P5nHEydyptgc3NTuGD8lUngonZidx//pNmfuIo/VcShxAEPhmLhc2icdLbH5w9CYc4G+zOmf7Mege7cCokDCRs8kD2+r5Pis985fPVa1UBjteJOnpm05SXPVGRjFB/ZZ5G5AJEe6LJSWKPQx4up02k1tMaN5b7NedDcvIDAY5eDc4iosAmsFvLMG9UQkfvesc02EbWDfH+Q2jLq4cO2cL4z2G+dW2BHnsdcJrPOo7MFvt1oGPWqrmIzPxPGPzJDDoA//sy3We2iHj3RQQLVwpXGYoIfEaXugHnTP5ZLNMVOA+eNiKnFAd9WUCQP1ESx/v8ZtRyk2ZxwDhxOAOag+JpE4VmQAqxO2MfmDZgcCjxrwVHWIT7zoxQ/z71fPGSxsbk8vg5UmkwnvC23LJKFsSpLTMrc02n/Px1fOK6zv86ik0ZHU98KRLnL3cEHJIaMQplX3sI8N3GKU4TPOz56dGx2r0Oa4BNnLvu/T5QMII7zHlPeDkH4y7ezsottkD8hWM6XR/kPddhyY7kzKkc7zeH8azpxoN5oDOZtKX/USJx6rqZNLQZUh6HmgJYM/TlLdJaYUqeOlKiPv2j3SadXiz76UUFtls1+7Oc3/RVwIv92HTd185AcJ4uFI5hzWzhpmtSHBBx8IE+/PCCjW1C03U7lIGvgv3i9ZvX786ef/B6RqSbYu+PPPEHRM6nU/LnpvuMyUgx3B2PuH9y15Ho3shc984igTgEVU79aXJfe04bWaOwbG9nsZSesYR3rYoo2sGxjUCLbldNMxFsduE3SpASZ+m9L8KnUhAcIKEkfu0rj0l107bEwXhgyAfgvCaHjpPSF0pnkdZogzKNzMnLm0/9Z1+fTJAEeHbyFf9Mv4Kbep3KG/cpIgAUBFBqpYlHxh4ff0ODkMpSPyejYynQQyvttge5Ewrr+S8L4F5Ld97buAv/Awcxfyyq9CPs+DB7hUyKC5L5OqcyiK6ZAR90YoW72lmWff1VBuUlEig8xOQSwH+Ufpnxx346Bovyp97jxjZnIGcucjYnPOQE3AylnSJtFdbq9A5xgigvqp/SBhrM7XI4M8Khe520GRxuMC8WsMWYZlUEJ4dL8YLyoILvz13NSbYKJ/c5wBkJgmH6xg4W4QwpZvYOBkPB/fCB1no4LMIBhNuWk4jHAy6ocFk4sNe22hSbVkzQqYH0gEBHuna9gRBC+mFKrEUBQKVpXv74y8HAHaP1wtQiD3bXyzQ4oJN76+2gdqtyCwgJxOm1e3xygg3f6eTeQkjQifLEEzocYyS6nXAWQ1GJFnHQCyCA+N7ntyJU4c3nhfr2b9+fvQnePX+r61dkkqQR0mDgyCZYTslogvjGy9HXHKQrlpCP1Hb9NMjCaiXpa325SRL9+Ti4PQqmKG1RkK0ylSFo1BucmU/vZHP4oj8bmnGIqYGm3q1aGUGFKl34K0b9/PTwjcJvd+gIvhh6rr7S2GPh4mDfeqhjoGXYq0C+XKXCaQheDlVGHWasROWVcdPA/R8lpSr0k6A0yJIcUrdN5Hu7qDl2rp1Gq/eROe61OlJWHHMGB8O0+skDXfs0e2a5sXTkiKBWytLg9Wexyool1WhBc9NkJRK/5NW4NAMVinl/tjPxULcUUw4rBFCSZh3ecuLfF8ZQdBmliSpUwPMGRk5r+9lqzc9zrrjK9yk8xfXUp7lem/w0xfohI7uFGG1ENOAA8A63tbRek4y6XvuDHu6NJElcz6jNZszgxTT4RbX5r98+/+Hs3dnF0V+Dn484+gxcOKFBsqiTJbCButINmEe5fy7/tZGsJVy8fa+oyFpIUxRaBA1vmKju0fHXU2sXLfOI2WfWm0ArBXv13SnaWbZp6q1XnbSjr/+NhSbRv7PAk6PjR9aXRCPrGjv0Oyvc7dRbaysigp+fBiwj/qPtnR8v/FLzo17peL+nC0UoTYFBGxLmQSJDnB6yJTvK/MaNHxNtOwTa7dQjUCcIg5+Pg7+j+f+NLY6mZG9+Gl/YRpCKAv5M9WAq/MexeFsnr5UsMhlAStAGCPaF61qJKi0m27TV2i2p7gNi1QoZcE+fh1Eeu3a930HNm8DP75A5IQ8WJmhsvBTNCCa0+5hb8oaH76laUITruvEsXZ2OMnbb13bumf7dZ3dbtTcMwnqHE4tDs5aIwm5ne/PN+wAT+0E6wNrmkgUAo6w3UohgjD0Tiu/EtC2FQfmSKiUyqk4RF5StUgqyyWvEDMtmRgcDof1Kyo8yMOrWzEksqXtgLn7a5zQyW1B6h8q2O/t3jxuGL4Mp+4ay4mSJ8jnwt1WOdyVlKV68ozrBhms0agqhU+h7aMgk6a3s28ZsATAeVoa2f067qe1+/T42avwRUdUAhWpYO1WjkexQlebYHPpMz6C2TwEQloFczaB9W3LuSAN32+Wrb7pEhbIbXCkHdTvtinwUN8F5ClYVVZPlxEuxa3v1JpYCV2BLskFt/8AdojZyO5rCZYZATiJYZkV0hfOtXUWxfaB1t8ZAYRZumiIC77q68IKDP4YIiCUpfGeaciN5LZXrVnbjwFGi/6jwknQaWNlFVyDpDf2wAcO1gMuGw1VMOLfmaElAqZiRBWlIGsvx9JnaLbVpLqHlcUkRcoix6z04opEocCubBzttypiiX7t9dtnWJs1jKBAReP5+M9hNfDlTEFJVq/An9rYH1M25V/ddy1Ya/kTA1nZzDsAcHOxwdRsJ0Jm8xrbDKSoVIBBS88N2guJZ/YqHrbrYoRUDsyM8gSVCOGrjWi+CU1yKFx+t3e38AVwyAeduBV27GPoBUVFux2XefkdhbOWWCgA7+T+AAc+5zTXnwX1EZuiwaRXExP27ob4B/bpIn8XcoOWe0QNyW6NZG3STZ9wlpmS+UdmMkx6j5YZp4gl1kwWIzq4ywV7SrZszRJNA5zd41Fyp5TDLHh2/wGGkr0wRy9HtCl4bjqgiUODSnYbY4WntilvrQ1YBSdeh5Lvu1xbftq/os+is2HZu4z7/honhkI5MYRAZm6Tva49OZeH+6FSDMvl2ygfNVEJG055OCEcYFALw/ueL+zZiXyvn3kyYqZIQ7mmpI6WiZ6o3L5KJ1dNXewzZQYAdbuK/Nil8EZbjxEcQPgr+LjRW/L8FWFtqroVle3jxjPzB8zhcu/PyIXTZ9+jDXRBZZplVj7KUlLilf2lbGqmfJdKDNyGfDhUOdFCp54BZlSHWWgJK2q+6chyzaMlIIRAPz/LIbAqnLPoLH82/tI1V1rNLjKTWAqBvRU407Y0gB8P1oUHwjkJjOdKD1lmbO0Cw2K5ShcGAC4laqtYgIJGOL8Acn+jmhf6Ax/sd+0Brj10l2iVvUAt0x6x6vyCa3snydHoc32tT9w7znPrPknte3B3+UW+9fM7hMJtjY4Bh3/KWI6BLl1V2cbE4xOIKDJzwuqejAxQzkXkEgYq6P1nSg6K4z20BqTnX62M0blLwlF/aVw6sFXDjdzrWZe7RjFocywpXPW3r30J0b7RcYcyl7hba3UjL+4ZwmhvJtOiYxAQqtXhyMqqkzuAEOZqmiocMgR/LrahRFmep/VG8WBN3E3hKTxQrqgnAD30oVv3SJsJWm0CIT/Ci9HFQ+teYOPtcrT36Ww1u07GEB31qk69ga2+fWu8p8zF1bXs9rVLuLHUtgSwXINcm9q47EnB1Q3b6HzoPxtl60GAa9SwMf2gvSgWIXe1smMV5JhdDtixV/WLdKl/k9TYT9FaFSMALpjpFjEwqli53Mbeoklt+txKxdC9sIqyrFrA7f7AvhAHNqrhFJqqRbMdSZg9xtdg2Y1UtkIs6dAYv+AqztEq5OGpSX+GqdaWD1bhkkUm+kWZY67wFzfP/9RWKX1QVBDlhoM2xyidVVKJGqQfY+YS9p0lRr66ofmYMjJur1dezqy6mQvp1oi+SqKM8s65wVKQWSKVTMSOB1iXLPc7WCUCvI6cib39zFKzfIdWlbpR//hCYBL8jF1lJFxZ4SaHGJS4QzY5ONIvQLVOqfvNpV3SSm2QIf0Os2PoMKChcpcPpqh4zeMnIdPJNIuZt1Y8mUCUSyJig8gPlSFPOoau52W5QtiLfPTN3Bbk4HHeaLlM6zh+RJafe89PDIxg6/Hh0uvD6wV2X0f8O4zzx3/rtW1TZpL3yirVPVV+Do0uTf0nxVcp8smOKMhVXr3G+Xpjvh0yi7kOPHdSJBiivjbOuUTuP9LraTya6HW1sz43+CwL6vLhhRAVx0VbtFcURM7rjhatL/YbHK2OxNREnkgI84OJTkPBIlJyVNBUVikVBuIk+GSAtf8MAITeqVAdPtv0imO2DnKNOFBr4PpqyI+nMmTv+mqjvJ+IDdMREUGr2kXUwZTGbOa4zc2o5FoNsN4+dTW2dctfd3SS9ygltDjpYMVPHENzR10oHW2Pm7Y3Z2a128J59HIXCl9ODdt0tjPeG8yamj6JI2+FD22EILDlqO70a6QTGJYO3JyJ666qQdGkhYLsZBNUPIV3gljgbeEbJW38z59sFjtOfxDEput58LStR6faArfZsqNfeYrTgIHKtMiaAc3evGkbunmHX55oQYHiunFQr6R0BLRjXuOGZKqgzp7imP2OgU8MzBcMZYubZN1M6H/cT5scdGlPITFLvau8fv+hg9RamymCiPetxtySar1rRrMy0cn46EVcLy0Z/APPhBBbC7cK6TVjMo4W6sLjLu3NMyVUOfY7lz49axzb7akD6+Ovje66/EnlPCeN7WzpTZlJXDaA4lsydMojSThHi4+wYBXfKZprRb2twDIKA72UVUVYP1ZbfnPxBvHxNRZ5wxGEqctXAxdn5heBk4TnUEFHlsIO1z3bq64ChCkiOzFNfYu/TDWrl+UoJZn13Vl9dDipzB/cB4Z3Gt8FyG9Cuz9F6Q38Sx/AS7mEpVu0CAb0zv9CVdWFTMydYgnC+mIxLSG6w5JT1rvbTjgcF3cRq5/r3LzpiDy7AxbdqrbZJPQc1cNetgC/gpvAf2LG5BR/p6mm196pw9JbmbSlDd9ZU58VDGR81K82ib4XXZAEpXWN+u/FbYkjK9mznGLigo0q/3Q03ivWWQ53a0nveEbt1VIfahmWrtzNwd1daCHsV1x5Q7T62EAz/6iEPiNN9RsGuMvf2y8MeHixuDSIDac/U/iR5/7htNJYueWRRny6Ahzcb765wJFpVXLYSyUWQ69g/IXrtaf3mf6h5wTEiVb0wt+X6Asbv8c5tMWilax10bVT6pqYb3sbOZTsZ/NGQSFV/mGbC90MCFTvjqlEu8UxmVDraydK3VLxL/rWWpHTdmY4eikFwF+Dlz+oPZSj/HXnYdwghhJvDF5L+rpGZ2FwZJ0RpfBaqlEz7t0sQv0BRFq42pcTGarchqlV1/tGTn+CF5AFfZ8kVwkMhHaWo5zf3Z4kEKtnRl5xm3SoeYvlAPwmV0NIxDRRj1McUaNNGGEjr07WrOC6S2REHd3+CG3OkNsd4RLVUs9X/gr2rvWRCfEhlcl7qY+3s8l8woB1r/LJMULV7QuF7coyekEX2E4UGhxPuKPmee7O2GYtxwWT4pnloTSXmhIFs+60JSfVpcfB/KQ4oh8FNAAA=", "s02_cv.py": "H4sIAEUo+GoC/8VYW4/jthV+968gWKAjIbYys0j74MIPg7QbBC02RbJ9KKYGwZGoMTMSqZKUdx3X/z3fISVb47Vn9q2LwVokzzk8l+9cJM75L0F17B1bMKc6JYOq5qxR8lk+qYX8JJ1ipbPeL7ay0ZUM2hqGv7BR7OPP9z9++PHDD6yTLmg6mc06Z4MtbcNunpztO1XdsCX7JTgw1lpVP9Dm39/bpsJ9xrKq7xpd4lIWydm3kBW0MiFdqjyrQesnYo3UW3XD2AuxSaIMTLdQG+pvVcMy2ONs1ZeQQupap5+0kc2MXfrn+8dWew8jbjwbr/sL672q2E8f/vFvFixrlfS9U6N7mDZ1Ez2Sz2Z/2yq3Y643xOKZTAYNHpz4zsPgwIJ8VobVzrZRNRiijTZPg7W1dUxJ1+yYD7brcDBn3iZK5UOkYgZGuqhCr0y0cRQC7tZW8EC5sbpUxezn3kAj6OGU71v52KglFKxB7TcwL4vU88iEixQhgK7ImfbMP+sOYSxms97D5iW81+3Chiwpne6C/9bfvhPltuh2bLGAkVI4awP76/3He2xEL/h4cQPatCxKv8WZ7cN4wP4zRGWxiMp4pupal4QEo4J4vB0OYOUnK3wg7y8WpKpnt+yO4Ls4YmRA3oxzPtNtZ12A8U8AqVfj2vrxKehWzcaF6VuYIT0z3bjVSVNhA39dNZvFiJW2bWH/QJB9b02tEaHHXjeVKGW5UeNicCwYuj4o0argdOlpozdBQCPZYlXrML8IymiIAHhERM0c0a7UZ2F0+QzerXRzIFVVusSRl1slfvXWpAgKQgfiRNi5KFuZrXbWtPCwAIgsIDyrVM1agCjLl5FHdmx1dF1x7556Iv8nrVyWDySFrCohh7OMTzDA5wjvf3sNFVcfXa+uciRUfDU5cANaKCsBnRUfIMSv0idEgcVg16/4NxPuB34GNL5+Vc5VMUdcviIg4vWlgLDr1EqbMBF1O2d3c/buuphYJC7y/ukqz5gcYEtVwUPnIVGwx2NZ5euJW8fDqyIj0kWl3TQYcfM6j2/tswK9LKkYrjjqm1MiINojDyGuKyLiiNMDZ/FgTFfryk3ciKnon1GLnSliiIVXjYqCR+pLjWd+3jeS/LJ+wtUplwds65rJImq8PKYQyNLWQFOpLdiiVgWe4diMl30lOXGnbVoW2gu5lbqh4pvlDHhUjJddP5jdOU0OShKW0aHbwfCqxgXniZ/J4phmSQJp31WFUxI1yG9BkLIqf3hAhlDcU0SFrvh6fZJc1UWr3JPKUMEoIol2Yz+teKPqMIYFvRj+BPEopDA2GJnlhWyaLB+lp9JeWRRqnKOYhHLDTk2BZgxqkf72TlQKzR9dg8cLgou6POCG1CFXK8ZjR+NrGOUVVSn4IKuc7SblQbdPHpyT2psF98AxRWzAGGyjfSD1KG6gFV7/hupsfUEUxa8WxU4WRyTPxzvzJH03Z0MDQ4whFuGDe9YFSnKvULlp7+TVYTuFDUWKJILx7DIc4JpyS1lQjGmZrgNli8EAbD4b+OdMfYYJwj5PjD7W+mzPy4hYvowWCkHNQAhcgPqOzfMqn+WHM+uP93AEJtIUJJk8kNIM0wTVLUCQsoEq2CkbYoYc+y6FbKwayxdNJ4Y0KPLGpZTMTGpwfiWLWNzQwzZ9XTcqGo2+gCZsW6qvQa1Ih/xL8RSjGO3xsgSkzHTFbwrDZNYok+1yYOEY1fwkhhLyK3T+v6g7UZOiUcciBpgDdJ+BQUW/OcVHYX5RUFUloT5/adEfXp9JtfG6Upfm0akQHYAVvaU6fTaeZMmpD0kxtJLd6ZHASYS1S5U/DSkv3YIxKFlEhHiIOUcPD7hzPT8utnL9km+cd4WRrUo4TS1/+cXkM9KeqC4QxXokqaDWfH+SfBAirvDr96T+QdR7ctCBXxRRUx6d5/8x3eiCbxgfku2iAGTXyBuLgM+iyPyyxpM2QjM7yyprVE69BFflV1lQP4I2vZpdl1fzFVJ7f8r0A/sf20Mq/SJqbE9YHcKXHyh+aWsIJLbia0vcG+B6uGLz2eyafQmT8d8wW2/Q8I4NIO5lp5BF4F1mD7eELwz/Bf2XXSbawOsEA4B7ct08dp2HweAI9NNzPBoMj0fH56shmBgUFR66/8WAUJdPI/+oUbwweXX9FnsL9rOXkWx3Yu6ucBV9V1FV2fN4J/rK1MVxc9jDiiKGVXqN5JQfWMWa9YYD+HFGXU66Cs2mAu7F5hRlcRuuHbZHpL15hxEExYFrwCJkpVcx6qKTN7Pk4bdlxmopWmQRZENGNoEVRp5wm+O7xp8x1nPCk3U70NDTm4JTgbXIhkZ24pMOm1F7SkuyAHPRWHZHCObsj2yyPQQ3z/PD5eii4dA8kb1apgTC8egLg3Etv4qTr5MEbUY5g/8vF4LjjNPOUynNXy1RcUotS7ZvH27w26PR7G7Wy+K7+oAxFF+TFu/v4mFciPpuPOSvRKHm9//6fsIl+3Jky2j3ReTp5I5u0ya/Ut3oo0wE1bV6j8wt6LUuzlLxPeJ6sZ+8Xai2C7th/M2Hefo095fx5XcYXl++B2BCjm8LV0bCZF8cvGmwp5jFxeq9xLxEnwygs4h1QIiosxD0AUGIQe/0NWH2O0ZQdkJmFAAA", "groups_ncc098_patients.csv": "...", "groups_ncc095_duponly.csv": "...", "groups_ncc098_duponly.csv": "..."}
for name, data in BUNDLE.items():
    open(f"{KIT}/{name}", "wb").write(gzip.decompress(base64.b64decode(data)))
DATA = None
for root, dirs, files in os.walk("/kaggle/input"):
    if {"training", "testing"} <= {d.lower() for d in dirs}:
        DATA = root; break
assert DATA, "Add the masoudnickparvar/brain-tumor-mri-dataset input first"
n = {s: sum(len(f) for _, _, f in os.walk(os.path.join(DATA, s))) for s in os.listdir(DATA)}
print("dataset:", DATA, n)
import torch
NGPU = torch.cuda.device_count()
print("GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)], "| torch", torch.__version__)
assert NGPU >= 1 or SMOKE, "Enable the GPU accelerator"

In [ ]:
CONFIGS = ["ncc098_patients", "ncc095_duponly", "ncc098_duponly"]
jobs = queue.Queue()
for seed in (0, 1, 2):            # seed 0 of every grouping first, then seeds 1 and 2
    for cfg in CONFIGS:
        jobs.put((cfg, seed))
def worker(gpu):
    while True:
        try: cfg, seed = jobs.get_nowait()
        except queue.Empty: return
        if hours() > TIME_BUDGET_HOURS - 1.2:
            print(f"SKIPPED (time budget): {cfg} seed {seed}", flush=True); continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        cmd = [sys.executable, "s02_cv.py", "--data_root", DATA, "--groups", f"groups_{cfg}.csv",
               "--out", f"{OUT}/{cfg}", "--models", "efficientnet_b0", "--modes", "two_stage",
               "--seeds", str(seed), "--protocol", "grouped", "--cache_dir", f"{WORK}/cache_gpu{gpu}"]
        if SMOKE: cmd.append("--smoke")
        print(f"[{hours():.2f} h] gpu{gpu} START {cfg} seed {seed}", flush=True)
        with open(f"{OUT}/log_{cfg}_s{seed}.txt", "w") as log:
            r = subprocess.run(cmd, cwd=KIT, env=env, stdout=log, stderr=subprocess.STDOUT)
        tail = open(f"{OUT}/log_{cfg}_s{seed}.txt").read().strip().splitlines()[-6:]
        print(f"[{hours():.2f} h] gpu{gpu} END {cfg} seed {seed} (exit {r.returncode})\n   " + "\n   ".join(tail), flush=True)
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(NGPU, 1))]
for t in threads: t.start()
for t in threads: t.join()
print(f"all jobs finished after {hours():.2f} h")

In [ ]:
import numpy as np, pandas as pd, shutil
rows = []
for cfg in CONFIGS:
    fs = sorted(glob.glob(f"{OUT}/{cfg}/cv/grouped/efficientnet_b0__two_stage__s*_f*.json"))
    r = [json.load(open(f)) for f in fs]
    if not r: continue
    def ms(k): v = np.array([x[k] for x in r]); return f"{v.mean():.4f} ± {v.std(ddof=1):.4f}" if len(v) > 1 else f"{v.mean():.4f}"
    rows.append({"grouping": cfg, "runs": len(r), "accuracy": ms("accuracy"), "balanced_acc": ms("balanced_accuracy"),
                 "macro_f1": ms("macro_f1"), "macro_auc": ms("macro_auc"), "ece": ms("ece"), "brier": ms("brier"),
                 "max_group_overlap": max(x["group_overlap_with_test"] for x in r)})
summary = pd.DataFrame(rows); summary.to_csv(f"{OUT}/summary.csv", index=False); print(summary.to_string(index=False))
for f in glob.glob(f"{KIT}/groups_*.csv"): shutil.copy(f, OUT)
Z = f"{WORK}/BPEX_R2_sensitivity_output.zip"
with zipfile.ZipFile(Z, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), WORK))
for d in glob.glob(f"{WORK}/cache_gpu*"): shutil.rmtree(d, ignore_errors=True)
print("DONE ->", Z, f"({os.path.getsize(Z)/1e6:.1f} MB) after {hours():.2f} h")